# R-Super on MSD Task07 Pancreas: reproduction and extension (BM 4302)

**Paper:** R-Super, *Learning Segmentation from Radiology Reports* (MICCAI 2025, arXiv 2507.05582).
**Code:** [github.com/MrGiovanni/R-Super](https://github.com/MrGiovanni/R-Super), pinned to commit `0cac36f`.

This notebook runs the **authors' own code**. Training uses `rsuper_train/train_ddp.py`, the losses come from
`rsuper_train/training/losses_foundation.py` (Volume Loss = `volume_loss_basic`, Ball Loss = `ball_loss`), the data
loaders are `training/dataset/dim3/dataset_abdomenatlas{,_UFO}.py`, and inference is `predict_abdomenatlas.py`.
The only new code is (1) a converter from MSD into the repo's input format plus a synthetic-report generator
(`rsuper_msd.py`, written below), (2) config edits for a T4 GPU, and (3) the evaluation.

### Substitutions (state these in both presentations)
| Paper | This notebook | Consequence |
|---|---|---|
| UCSF CT-report pairs (private, 6.7k) | 150 MSD tumour CTs used as **report-only** (their masks are hidden from training) | Far smaller report set |
| LLM-extracted report fields (count, location, size) | **Synthetic reports**: tumour count, head/body/tail location and two axial diameters measured on the MSD mask, then log-normal noise (sigma = 0.10) | The reports are proxies. No LLM extraction error, and the noise is controlled |
| AbdomenAtlas 2.0/3.0 masks (thousands) | 50 MSD CTs with masks ("few masks" setting) | |
| Organ / sub-segment masks from nnU-Net | MSD ground-truth pancreas; head/body/tail = thirds along the pancreas principal axis | Optimistic organ masks; the sub-segments are approximate |
| 128^3 patches, A100s, 100 epochs x 1000 it | 96^3 patches, 2x T4, iterations cut down (see `ITERS_PER_EPOCH` and the epoch settings) | Under-trained compared with the paper |
| Detection AUC with healthy CTs | MSD has no healthy CTs, so lesion-level F1 and patient-level sensitivity only. Specificity needs normals (TCIA Pancreas-CT, optional) | |

### How to run on Kaggle
1. **Settings:** *Internet* ON; *Persistence* = **Files only**, so `/kaggle/working` survives between sessions; add the secret `HF_TOKEN`.
2. **Stage A (CPU session, accelerator None):** `RUN_PREP=True`. This downloads MSD (12 GB) into `/kaggle/temp` and writes about 11-13 GB of npz to `/kaggle/working/rsuper_data`.
3. **Stage B (GPU T4 x2):** set `RUN_PREP=False`, then run `RUN_PROBE=True` once to measure speed and memory. Then run `RUN_BASELINE`, then `RUN_FINETUNE`, then `RUN_EVAL`.
   For long runs use *Save Version → Save & Run All*, which keeps running for up to 12 h without the browser. Training resumes automatically from `fold_0_latest.pth`.

In [1]:
# ======================= CONFIG: the only cell you normally edit =======================
import os, sys, json, glob, time, shutil, subprocess, math, threading

# ---- stage switches ----
RUN_PREP     = False     # CPU session: download MSD, convert to the repo's npz format, write synthetic reports
RUN_QC       = True     # quick checks/plots of the converted data
RUN_PROBE    = False    # GPU: one short epoch of each training stage -> s/iter + GPU memory
RUN_BASELINE = True    # GPU: masks-only model (50 masks)  [paper: "standard segmentation" baseline]
RUN_FINETUNE = []       # GPU: any of ['control', 'volume', 'volume_ball'] (each starts from the baseline)
RUN_EVAL     = []       # GPU: any of ['baseline', 'control', 'volume', 'volume_ball']
PUSH_DATA_TO_HF = False # optional: back up rsuper_data to a PRIVATE HF dataset repo (see the persistence cell)

# ---- paths ----
WORK    = '/kaggle/working'
TMP     = '/kaggle/temp'                       # large, not persisted; holds the 12 GB MSD zip
REPO_DIR, REPO_COMMIT = f'{WORK}/R-Super', '0cac36f'
RS      = f'{REPO_DIR}/rsuper_train'           # all repo scripts run with this as cwd
EXP     = f'{WORK}/exp/'                       # checkpoints: {EXP}{dataset}/{unique_name}/fold_0_latest.pth
RESULTS = f'{WORK}/results'
LOGS    = f'{WORK}/logs'

def find_data_root():
    # rsuper_data from this session, or attached as an input (an earlier version's output or a dataset)
    for p in [f'{WORK}/rsuper_data'] + sorted(glob.glob('/kaggle/input/**/rsuper_data', recursive=True)):
        if os.path.exists(f'{p}/meta/splits.json'):
            return p
    return f'{WORK}/rsuper_data'
DATA = find_data_root()
MASK_NPZ, REPORT_NPZ, EVAL_NPZ, META = (f'{DATA}/{d}' for d in ['mask_npz', 'report_npz', 'eval_npz', 'meta'])

# ---- data / reports ----
SEED = 42
SPLIT_SIZES = [('test', 50), ('val', 15), ('mask_train', 50), ('report_only', 150)]   # tumour cases
SIGMAS = [0.0, 0.10, 0.20, 0.30]      # report noise levels written to disk (the extension sweep uses these)
TRAIN_SIGMA = 0.10                    # report noise used for the main experiments
MIN_TUMOR_VOX = 50                    # components < 50 mm^3 (~4.6 mm sphere) are not "reported"
TRAIN_MARGIN = 48                     # mm kept around the pancreas for training cases
PREP_WORKERS = max(1, (os.cpu_count() or 4))

# ---- training (T4-sized; the paper uses 128^3, 1000 it/epoch, 100+ epochs) ----
CROP = 96
ITERS_PER_EPOCH = 200
BASE_EPOCHS, BASE_LR = 50, 1e-3       # README: masks-only stage uses lr 0.001
FT_EPOCHS, FT_LR = 25, 1e-4           # README: report stage uses lr 0.0001, starting from the masks-only model
REPORT_W = 0.1                        # --report_volume_loss_basic (paper/README value)
VOLUME_TOL = 0.1                      # --volume_loss_tolerance (paper: 10 %; the argparse default is 0.2)
GPUS, BATCH, WORKERS = '0,1', 2, 2    # batch is global (1 per T4)
MAX_RESTARTS = 3

for d in [TMP, EXP, RESULTS, LOGS]:
    os.makedirs(d, exist_ok=True)
print('DATA =', DATA)

DATA = /kaggle/working/rsuper_data


## 0. Environment: clone the authors' repo (pinned) and install its dependencies

In [2]:
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/MrGiovanni/R-Super.git', REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-q', REPO_COMMIT], check=True)
print(subprocess.run(['git', '-C', REPO_DIR, 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)

# rsuper_train/requirements.txt pins torch 2.1; we keep Kaggle's torch and add what the imported modules need
# (model/dim3/__init__.py imports UNETR/SwinUNETR -> monai, nnFormer -> batchgenerators + timm; MedFormer -> einops).
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'einops', 'monai', 'batchgenerators', 'timm',
                'SimpleITK', 'nibabel', 'scikit-image', 'huggingface_hub'], check=True)
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| GPUs', torch.cuda.device_count())
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout
      if shutil.which('nvidia-smi') else 'no GPU (fine for RUN_PREP)')
print(subprocess.run(['df', '-h', WORK, TMP], capture_output=True, text=True).stdout)

0cac36f Update README.md

torch 2.11.0+cpu | CUDA False | GPUs 0
no GPU (fine for RUN_PREP)
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop2       20G   48M   20G   1% /kaggle/working
overlay         8.0T  5.6T  2.5T  70% /



## 1. Data preparation (CPU session)

### 1a. Download MSD Task07 Pancreas
The source is the public HF mirror `Luffy503/VoCo_Downstream`, file `MSD_Decathlon/Dataset007_Pancreas.zip` (nnU-Net layout, 281 labelled CTs:
label 1 = pancreas, 2 = pancreatic mass). The converter reads the zip directly, so the 12 GB are not extracted a second time.

In [3]:
MSD_ZIP = f'{TMP}/msd/MSD_Decathlon/Dataset007_Pancreas.zip'
if RUN_PREP and not os.path.exists(MSD_ZIP):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception as e:
        print('HF_TOKEN secret not available (public download may still work):', e)
    from huggingface_hub import hf_hub_download
    MSD_ZIP = hf_hub_download('Luffy503/VoCo_Downstream', filename='MSD_Decathlon/Dataset007_Pancreas.zip',
                              repo_type='dataset', local_dir=f'{TMP}/msd')
if os.path.exists(MSD_ZIP):
    print(MSD_ZIP, round(os.path.getsize(MSD_ZIP) / 1e9, 2), 'GB')

### 1b. Converter and synthetic-report generator (`rsuper_msd.py`)
For the code review, here is how this module maps to the paper and repo:
* **Repo format.** The repo's `dataset_conversion/abdomenatlas_3d.py` and `nii2npz.py` produce `BDMAP_xxxxxxxx.npz` (CT, float32, 1 mm, clipped to [-991, 500] HU and z-scored)
  and `BDMAP_xxxxxxxx_gt.npz` (int8, one channel per class, classes sorted alphabetically). We import the repo's own `reorient_image / ResampleXYZAxis / ResampleLabelToRef / pad`.
* **Why 1 mm matters.** Both report losses take sizes in voxels and assume 1 mm voxels: `estimate_tumor_volume` turns the report's mm into mm^3 and uses that directly as a voxel count.
* **Report-only cases** (`report_npz/`) get organ channels only (`pancreas`, `pancreas_head/body/tail`) and **no** `pancreatic_lesion` channel.
  The repo's `assign_labels()` then marks the tumour channel inside the organ sub-segments named by the report as **unknown** (`unk_voxels`), so the standard BCE+Dice loss skips those voxels.
  Only the Volume and Ball losses supervise them.
* **Synthetic report.** For each tumour (a 26-connected component of at least 50 mm^3) we record its location (the head/body/tail thirds that hold at least 15 % of its voxels) and two diameters measured on the axial slice where the tumour is largest.
  A report is written as `"d1 x d2"` mm. The repo then sets `d3 = (d1+d2)/2` and `V = 4/3*pi*(d1/2)(d2/2)(d3/2)` (`estimate_tumor_volume`).
  The noise is `d_report = round(d_true * exp(sigma * z))`, with `z ~ N(0,1)` drawn once per tumour, so every sigma level reuses the same draws.

In [4]:
%%writefile /kaggle/working/rsuper_msd.py
"""
rsuper_msd.py - MSD Task07 Pancreas -> R-Super training format + synthetic radiology reports.

Course-project substitution (documented in the notebook):
  * Paper: UCSF CT-report pairs (private) + AbdomenAtlas masks.  Here: MSD Task07 Pancreas only.
  * "Reports" are SYNTHETIC: tumour count, location (head/body/tail) and two axial diameters are
    measured on the MSD tumour mask and perturbed with log-normal noise. They stand in for the
    LLM-extracted report fields the paper uses (R-Super report_extraction/ -> per-tumour CSV).

What is copied from the repo (imported, not re-written):
  * dataset_conversion/utils.py      : reorient_image, ResampleXYZAxis, ResampleLabelToRef
  * dataset_conversion/nii2npz.py    : pad()  (pad every axis to >= 128)
  and the same steps as abdomenatlas_3d.py + nii2npz.py:
      reorient RAI -> 1x1x1 mm (BSpline in-plane, nearest along z; nearest for labels)
      -> clip HU [-991, 500] -> z-score -> pad -> np.savez_compressed (key 'arr_0')
      image: float32 (D,H,W);  labels: int8 (C,D,H,W), one channel per class, classes sorted by name.

Deviations (all disclosed):
  1. Before normalisation each CT is cropped to the body bounding box (HU > -500); no label is used.
  2. Training cases are then cropped to the pancreas bounding box + margin to save disk.  The
     repo's loader samples 96^3/128^3 patches around organs/tumours anyway.  Eval cases keep the
     whole body crop (no label is used to crop them).
  3. MSD has no pancreas head/body/tail masks.  We split the pancreas mask in thirds along its
     principal axis (head = patient-right end).  The paper gets these from an nnU-Net sub-segment model.
  4. Organ masks for report-only cases are the MSD ground-truth pancreas (label >= 1), not predicted.
"""
import os
import json
import math
import zipfile
import tempfile
import importlib.util

import numpy as np
import SimpleITK as sitk
from scipy import ndimage as ndi

MASK_CLASSES = ['pancreas', 'pancreas_body', 'pancreas_head', 'pancreas_tail', 'pancreatic_lesion']
REPORT_CLASSES = ['pancreas', 'pancreas_body', 'pancreas_head', 'pancreas_tail']   # NO lesion channel
SEGMENTS = ['head', 'body', 'tail']
SPLIT_DIRS = {'mask_train': 'mask_npz', 'report_only': 'report_npz', 'val': 'eval_npz', 'test': 'eval_npz'}

# Columns of the repo's per-tumour report table (rsuper_train/Merlin_per_tumor_metadata_hf_clean.csv),
# which training/dataset/dim3/dataset_abdomenatlas_UFO.py reads.
REPORT_COLUMNS = ['Encrypted Accession Number', 'Tumor ID', 'Organ', 'Tumor Type', 'Tumor Location',
                  'Tumor Size (mm)', 'Type Certainty', 'BDMAP ID', 'no lesion', 'Standardized Organ',
                  'Standardized Location', 'Unknow Tumor Size']


# ----------------------------------------------------------------------------- repo code
_REPO = {}


def _load_module(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


def repo_fns(repo_train_dir):
    """Import the repo's own conversion helpers (unchanged) from rsuper_train/dataset_conversion."""
    if 'utils' not in _REPO:
        dc = os.path.join(repo_train_dir, 'dataset_conversion')
        _REPO['utils'] = _load_module(os.path.join(dc, 'utils.py'), 'rsuper_dc_utils')
        _REPO['nii2npz'] = _load_module(os.path.join(dc, 'nii2npz.py'), 'rsuper_dc_nii2npz')
    return _REPO['utils'], _REPO['nii2npz']


# ----------------------------------------------------------------------------- MSD source
def bdmap_id(case):
    """pancreas_001 -> BDMAP_00000001 (the repo slices IDs as f[:len('BDMAP_00000000')])."""
    return 'BDMAP_%08d' % int(case.split('_')[-1])


class MSDSource:
    """Reads MSD Task07 cases from the nnU-Net style zip (Dataset007_Pancreas.zip) or an extracted folder."""

    def __init__(self, src):
        self.src = src
        self.is_zip = src.endswith('.zip')
        if not self.is_zip:
            inner = os.path.join(src, 'Dataset007_Pancreas')
            self.root = inner if os.path.isdir(inner) else src

    def _rel(self, case, kind):
        if kind == 'image':
            return 'Dataset007_Pancreas/imagesTr/%s_0000.nii.gz' % case
        return 'Dataset007_Pancreas/labelsTr/%s.nii.gz' % case

    def list_cases(self):
        if self.is_zip:
            with zipfile.ZipFile(self.src) as z:
                names = z.namelist()
        else:
            names = ['Dataset007_Pancreas/imagesTr/' + f for f in os.listdir(os.path.join(self.root, 'imagesTr'))]
        cases = [os.path.basename(n)[:-len('_0000.nii.gz')] for n in names
                 if '/imagesTr/' in n and n.endswith('_0000.nii.gz')]
        return sorted(cases)

    def read(self, case, kind):
        rel = self._rel(case, kind)
        if not self.is_zip:
            return _read_sitk(os.path.join(self.root, rel.split('/', 1)[1]))
        with zipfile.ZipFile(self.src) as z:
            data = z.read(rel)
        fd, tmp = tempfile.mkstemp(suffix='.nii.gz')
        try:
            with os.fdopen(fd, 'wb') as f:
                f.write(data)
            return _read_sitk(tmp)
        finally:
            os.remove(tmp)


def _read_sitk(path):
    """sitk.ReadImage with the repo's nibabel fallback for non-orthonormal direction cosines
    (same idea as read_sitk_with_nib_fallback in dataset_conversion/abdomenatlas_3d.py)."""
    try:
        return sitk.ReadImage(path)
    except RuntimeError as e:
        if 'orthonormal' not in str(e).lower():
            raise
        import nibabel as nib
        img = nib.load(path)
        aff = img.affine.copy()
        u, _, vt = np.linalg.svd(aff[:3, :3])
        aff[:3, :3] = u @ vt
        fd, tmp = tempfile.mkstemp(suffix='.nii.gz')
        os.close(fd)
        try:
            nib.Nifti1Image(np.asanyarray(img.dataobj), aff, img.header).to_filename(tmp)
            return sitk.ReadImage(tmp)
        finally:
            os.remove(tmp)


def scan_case(args):
    """Label-only pass: voxel counts, used to decide which cases contain tumour before splitting."""
    src, case = args
    lab = sitk.GetArrayFromImage(MSDSource(src).read(case, 'label'))
    return case, int((lab >= 1).sum()), int((lab == 2).sum())


def make_splits(tumor_cases, sizes, seed=42):
    """Deterministic split of tumour cases (sorted, then permuted with numpy default_rng(seed))."""
    cases = sorted(tumor_cases)
    perm = np.random.default_rng(seed).permutation(len(cases))
    cases = [cases[i] for i in perm]
    out, i = {}, 0
    for name, n in sizes:
        out[name] = sorted(cases[i:i + n])
        i += n
    out['unused'] = sorted(cases[i:])
    return out


# ----------------------------------------------------------------------------- geometry
def resample_1mm(img, lab, U):
    """Exactly the resampling of ResampleImage() in dataset_conversion/abdomenatlas_3d.py."""
    img = U.reorient_image(img, 'RAI')
    lab = U.reorient_image(lab, 'RAI')
    if lab.GetSize() != img.GetSize():
        raise ValueError('image/label size mismatch %s vs %s' % (img.GetSize(), lab.GetSize()))
    if (np.allclose(lab.GetOrigin(), img.GetOrigin(), atol=1e-2)
            and np.allclose(lab.GetDirection(), img.GetDirection(), atol=1e-3)
            and np.allclose(lab.GetSpacing(), img.GetSpacing(), atol=1e-3)):
        lab.CopyInformation(img)  # remove float noise in headers before nearest-neighbour resampling
    sp = img.GetSpacing()
    img_xy = U.ResampleXYZAxis(img, space=(1.0, 1.0, sp[2]), interp=sitk.sitkBSpline)
    lab_xy = U.ResampleLabelToRef(lab, img_xy, interp=sitk.sitkNearestNeighbor)
    img_xyz = U.ResampleXYZAxis(img_xy, space=(1.0, 1.0, 1.0), interp=sitk.sitkNearestNeighbor)
    lab_xyz = U.ResampleLabelToRef(lab_xy, img_xyz, interp=sitk.sitkNearestNeighbor)
    return img_xyz, lab_xyz


def body_bbox(ct, thr=-500.0, margin=5, step=4):
    """Bounding box (z,y,x) of the largest connected component with HU > thr. Uses no label."""
    small = ct[::step, ::step, ::step] > thr
    cc, n = ndi.label(small)
    if n == 0:
        return [(0, s) for s in ct.shape]
    k = int(np.argmax(ndi.sum(small, cc, range(1, n + 1)))) + 1
    sl = ndi.find_objects((cc == k).astype(np.int32))[0]
    return [(max(0, s.start * step - margin), min(dim, s.stop * step + margin)) for s, dim in zip(sl, ct.shape)]


def mask_bbox(mask, margin, shape):
    idx = np.argwhere(mask)
    lo, hi = idx.min(0), idx.max(0) + 1
    return [(max(0, int(a) - margin), min(int(d), int(b) + margin)) for a, b, d in zip(lo, hi, shape)]


def crop(arr, bb):
    return arr[..., bb[0][0]:bb[0][1], bb[1][0]:bb[1][1], bb[2][0]:bb[2][1]]


def split_pancreas(panc, origin, direction):
    """Approximate head/body/tail: thirds along the pancreas principal axis.
    ITK physical space is LPS (+x = patient left), so the axis is oriented towards +x:
    the patient-right end is the head, the left end is the tail."""
    zyx = np.argwhere(panc)
    masks = {s: np.zeros(panc.shape, bool) for s in SEGMENTS}
    if len(zyx) < 30:
        masks['body'][panc] = True
        return masks, None
    xyz = zyx[:, ::-1].astype(np.float64)                       # numpy (z,y,x) -> ITK (x,y,z)
    D = np.asarray(direction, dtype=np.float64).reshape(3, 3)
    phys = xyz @ D.T + np.asarray(origin, dtype=np.float64)     # spacing is 1 mm after resampling
    X = phys - phys.mean(0)
    rng = np.random.default_rng(0)
    sub = X[rng.choice(len(X), size=min(len(X), 20000), replace=False)]
    v = np.linalg.svd(sub, full_matrices=False)[2][0]
    if v[0] < 0:
        v = -v
    t = X @ v
    lo, hi = np.percentile(t, [1, 99])
    b1, b2 = lo + (hi - lo) / 3.0, lo + 2.0 * (hi - lo) / 3.0
    seg = np.where(t < b1, 0, np.where(t < b2, 1, 2))
    for k, s in enumerate(SEGMENTS):
        sel = zyx[seg == k]
        masks[s][sel[:, 0], sel[:, 1], sel[:, 2]] = True
    return masks, v.tolist()


def _extents(pts):
    """Extent (mm) of a point cloud along its principal axes, largest first (+1 voxel)."""
    if len(pts) < 2:
        return [1.0] * pts.shape[1]
    X = pts - pts.mean(0)
    vt = np.linalg.svd(X, full_matrices=False)[2]
    proj = X @ vt.T
    ext = proj.max(0) - proj.min(0) + 1.0
    return sorted(ext.tolist(), reverse=True)


def tumor_attributes(lesion, seg_masks, min_vox):
    """The simulated 'radiologist': reads the tumour mask and returns what a report would say
    (count, location, two axial diameters) plus true values kept only for analysis."""
    cc, _ = ndi.label(lesion, structure=np.ones((3, 3, 3), bool))
    tumors = []
    for i, sl in enumerate(ndi.find_objects(cc), 1):
        if sl is None:
            continue
        comp = cc[sl] == i
        vol = int(comp.sum())
        areas = comp.sum(axis=(1, 2))
        zk = int(np.argmax(areas))                            # axial slice with the largest cross-section
        d1, d2 = _extents(np.argwhere(comp[zk]).astype(np.float64))[:2] if areas[zk] > 1 else (1.0, 1.0)
        ext3 = _extents(np.argwhere(comp).astype(np.float64))
        frac = {s: float((comp & seg_masks[s][sl]).sum()) / vol for s in SEGMENTS}
        loc = [s for s in SEGMENTS if frac[s] >= 0.15] or [max(frac, key=frac.get)]
        cz, cy, cx = (np.argwhere(comp).mean(0) + [s.start for s in sl]).tolist()
        tumors.append(dict(
            true_volume_mm3=vol,
            d1_axial_mm=float(d1), d2_axial_mm=float(d2),
            pca3d_extents_mm=[float(e) for e in ext3],
            craniocaudal_mm=int(comp.any(axis=(1, 2)).sum()),
            location=' / '.join(loc), segment_fraction=frac,
            centroid_zyx=[cz, cy, cx]))
    tumors.sort(key=lambda t: -t['true_volume_mm3'])
    for t in tumors:
        t['reported'] = t['true_volume_mm3'] >= min_vox
    if tumors and not any(t['reported'] for t in tumors):
        tumors[0]['reported'] = True                          # every tumour case reports >= 1 tumour
    return tumors


# ----------------------------------------------------------------------------- conversion
def convert_case(job):
    """One MSD case -> <out>/<mask_npz|report_npz|eval_npz>/BDMAP_xxxxxxxx{.npz,_gt.npz} + attributes."""
    case, split, cfg = job['case'], job['split'], job['cfg']
    bd = bdmap_id(case)
    dst = os.path.join(cfg['out'], SPLIT_DIRS[split])
    cache = os.path.join(cfg['out'], 'meta', 'case_cache', bd + '.json')
    if os.path.exists(cache) and os.path.exists(os.path.join(dst, bd + '_gt.npz')):
        with open(cache) as f:
            return json.load(f)

    U, N = repo_fns(cfg['repo_train_dir'])
    src = MSDSource(cfg['src'])
    img0, lab0 = src.read(case, 'image'), src.read(case, 'label')
    img, lab = resample_1mm(img0, lab0, U)
    ct = sitk.GetArrayFromImage(img).astype(np.float32)        # (z,y,x), HU
    L = sitk.GetArrayFromImage(lab).astype(np.uint8)

    # 1) body crop (no label used) and index -> physical origin update
    bb = body_bbox(ct)
    D = np.asarray(img.GetDirection()).reshape(3, 3)
    origin = (np.asarray(img.GetOrigin()) + D @ np.array([bb[2][0], bb[1][0], bb[0][0]], float)).tolist()
    ct, L = crop(ct, bb), crop(L, bb)
    panc_lost = int((sitk.GetArrayFromImage(lab) >= 1).sum() - (L >= 1).sum())

    # 2) normalisation exactly as nii2npz.py (statistics over the body crop)
    ct = np.clip(ct, -991, 500)
    ct = (ct - ct.mean()) / ct.std()

    panc, lesion = L >= 1, L == 2          # pancreas mask includes the tumour (tumour lies inside the organ)
    segs, axis = split_pancreas(panc, origin, img.GetDirection())

    # 3) training cases: crop to pancreas bbox + margin (disk); eval cases keep the body crop
    if split in ('mask_train', 'report_only'):
        bb2 = mask_bbox(panc, cfg['train_margin'], panc.shape)
        ct, panc, lesion = crop(ct, bb2), crop(panc, bb2), crop(lesion, bb2)
        segs = {k: crop(v, bb2) for k, v in segs.items()}

    # 4) simulated radiologist reads the tumour mask -> report attributes.
    #    For report_only cases THIS IS THE ONLY USE OF THE TUMOUR MASK; it is not written to disk below.
    tumors = tumor_attributes(lesion, segs, cfg['min_tumor_vox'])

    channels = {'pancreas': panc, 'pancreas_head': segs['head'], 'pancreas_body': segs['body'],
                'pancreas_tail': segs['tail']}
    if split != 'report_only':
        channels['pancreatic_lesion'] = lesion
    names = sorted(channels)
    assert names == (REPORT_CLASSES if split == 'report_only' else MASK_CLASSES)
    lab_arr = np.stack([channels[n] for n in names]).astype(np.int8)
    ct, lab_arr = N.pad(ct, lab_arr)

    os.makedirs(dst, exist_ok=True)
    np.savez_compressed(os.path.join(dst, bd + '.npz'), ct.astype(np.float32))
    np.savez_compressed(os.path.join(dst, bd + '_gt.npz'), lab_arr.astype(np.int8))

    rec = dict(case=case, bdmap=bd, split=split, folder=SPLIT_DIRS[split],
               orig_spacing=list(img0.GetSpacing()), orig_size=list(img0.GetSize()),
               npz_shape=list(ct.shape), channels=names, pancreas_axis_lps=axis,
               pancreas_vox=int(panc.sum()), pancreas_vox_lost_by_body_crop=panc_lost,
               lesion_vox=int(lesion.sum()), tumors=tumors)
    os.makedirs(os.path.dirname(cache), exist_ok=True)
    with open(cache, 'w') as f:
        json.dump(rec, f)
    return rec


def write_lists(out, splits, records):
    """list/dataset.yaml and list/label_names.yaml, as written by abdomenatlas_3d.py / nii2npz.py."""
    import yaml
    for folder, classes in [('mask_npz', MASK_CLASSES), ('report_npz', REPORT_CLASSES), ('eval_npz', MASK_CLASSES)]:
        ids = sorted(r['bdmap'] for r in records if r['folder'] == folder)
        os.makedirs(os.path.join(out, folder, 'list'), exist_ok=True)
        with open(os.path.join(out, folder, 'list', 'dataset.yaml'), 'w') as f:
            yaml.dump(ids, f)
        with open(os.path.join(out, folder, 'list', 'label_names.yaml'), 'w') as f:
            yaml.dump(list(classes), f)


# ----------------------------------------------------------------------------- reports
def add_noise_draws(records, seed=42):
    """Fix one pair of standard-normal draws per reported tumour, so that reports at different
    sigma differ only in noise magnitude: d_report = round(d_true * exp(sigma * z))."""
    rng = np.random.default_rng(seed)
    for r in sorted(records, key=lambda r: r['bdmap']):
        for t in r['tumors']:
            t['noise_z'] = rng.standard_normal(2).tolist()
    return records


def report_rows(records, sigma, ids=None):
    """Rows in the repo's per-tumour report format. Size string 'd1 x d2' (mm): the repo sets
    d3 = (d1+d2)/2 and volume = 4/3*pi*(d1/2)(d2/2)(d3/2) (estimate_tumor_volume)."""
    rows, analysis = [], []
    for r in sorted(records, key=lambda r: r['bdmap']):
        if ids is not None and r['bdmap'] not in ids:
            continue
        reported = [t for t in r['tumors'] if t['reported']]
        if not reported:
            rows.append({'Encrypted Accession Number': 'SYN_' + r['bdmap'], 'Tumor ID': 'no lesion',
                         'Organ': 'no lesion', 'Tumor Type': 'no lesion', 'Tumor Location': 'no lesion',
                         'Tumor Size (mm)': 'U', 'Type Certainty': 'no lesion', 'BDMAP ID': r['bdmap'],
                         'no lesion': True, 'Standardized Organ': 'u', 'Standardized Location': '',
                         'Unknow Tumor Size': 'no'})
            continue
        for k, t in enumerate(reported, 1):
            z = t['noise_z']
            d1 = max(int(round(t['d1_axial_mm'] * math.exp(sigma * z[0]))), 3)
            d2 = max(int(round(t['d2_axial_mm'] * math.exp(sigma * z[1]))), 3)
            d1, d2 = max(d1, d2), min(d1, d2)
            size = '%.1f x %.1f' % (d1, d2)
            rows.append({'Encrypted Accession Number': 'SYN_' + r['bdmap'], 'Tumor ID': 'tumor %d' % k,
                         'Organ': 'pancreas', 'Tumor Type': 'pancreatic mass (MSD label 2)',
                         'Tumor Location': t['location'], 'Tumor Size (mm)': size, 'Type Certainty': 'synthetic',
                         'BDMAP ID': r['bdmap'], 'no lesion': False, 'Standardized Organ': 'pancreas',
                         'Standardized Location': t['location'], 'Unknow Tumor Size': 'no'})
            d3 = (d1 + d2) / 2.0
            analysis.append(dict(bdmap=r['bdmap'], split=r['split'], tumor=k, sigma=sigma, size=size,
                                 location=t['location'], d1_true=t['d1_axial_mm'], d2_true=t['d2_axial_mm'],
                                 true_volume_mm3=t['true_volume_mm3'],
                                 report_volume_mm3=4.0 / 3.0 * math.pi * (d1 / 2) * (d2 / 2) * (d3 / 2)))
    return rows, analysis


def per_ct_metadata(records):
    """Per-CT table in the format the repo's evaluation scripts expect
    ('number of pancreatic lesion instances', see calculate_sensitivity_specificity.py)."""
    out = []
    for r in sorted(records, key=lambda r: r['bdmap']):
        rep = [t for t in r['tumors'] if t['reported']]
        out.append({'BDMAP ID': r['bdmap'], 'split': r['split'], 'msd_case': r['case'],
                    'number of pancreatic lesion instances': len(rep),
                    'largest pancreatic lesion diameter (cm)': (max(t['d1_axial_mm'] for t in rep) / 10.0) if rep else 0.0,
                    'largest pancreatic lesion location (head, body, tail)': rep[0]['location'] if rep else '',
                    'no lesion': int(len(rep) == 0)})
    return out

Writing /kaggle/working/rsuper_msd.py


### 1c. Scan → split → convert → reports
The splits are deterministic (seed 42) and are saved to `meta/splits.json`. Once that file exists, it is reused and never recomputed.

In [5]:
if RUN_PREP:
    import importlib, numpy as np, pandas as pd
    from concurrent.futures import ProcessPoolExecutor
    from tqdm.auto import tqdm
    sys.path.insert(0, WORK)
    import rsuper_msd as rm; importlib.reload(rm)

    DATA = f'{WORK}/rsuper_data'
    MASK_NPZ, REPORT_NPZ, EVAL_NPZ, META = (f'{DATA}/{d}' for d in ['mask_npz', 'report_npz', 'eval_npz', 'meta'])
    os.makedirs(META, exist_ok=True)

    # --- 1) label scan: which cases contain tumour (MSD: expected all 281) ---
    if os.path.exists(f'{META}/splits.json'):
        splits = json.load(open(f'{META}/splits.json'))
        print('re-using existing splits')
    else:
        cases = rm.MSDSource(MSD_ZIP).list_cases()
        with ProcessPoolExecutor(PREP_WORKERS) as ex:
            scan = list(tqdm(ex.map(rm.scan_case, [(MSD_ZIP, c) for c in cases]), total=len(cases), desc='scan'))
        pd.DataFrame(scan, columns=['case', 'pancreas_vox', 'tumor_vox']).to_csv(f'{META}/label_scan.csv', index=False)
        tumor_cases = [c for c, p, t in scan if t > 0]
        print(len(cases), 'cases,', len(tumor_cases), 'with tumour')
        splits = rm.make_splits(tumor_cases, SPLIT_SIZES, seed=SEED)
        json.dump(splits, open(f'{META}/splits.json', 'w'), indent=1)
    print({k: len(v) for k, v in splits.items()})

    # --- 2) conversion (resumable: finished cases are cached in meta/case_cache) ---
    cfg = dict(out=DATA, src=MSD_ZIP, repo_train_dir=RS, train_margin=TRAIN_MARGIN, min_tumor_vox=MIN_TUMOR_VOX)
    jobs = [dict(case=c, split=s, cfg=cfg) for s in ['test', 'val', 'mask_train', 'report_only'] for c in splits[s]]
    t0 = time.time()
    records, failed = [], []
    with ProcessPoolExecutor(PREP_WORKERS) as ex:
        futs = {ex.submit(rm.convert_case, j): j['case'] for j in jobs}
        from concurrent.futures import as_completed
        for fu in tqdm(as_completed(futs), total=len(futs), desc='convert'):
            try:
                records.append(fu.result())
            except Exception as e:
                failed.append((futs[fu], repr(e)))
    print(f'converted {len(records)} cases in {(time.time()-t0)/60:.1f} min; failed: {failed}')
    rm.write_lists(DATA, splits, records)

    # --- 3) synthetic reports ---
    records = rm.add_noise_draws(records, seed=SEED)
    json.dump(records, open(f'{META}/clean_tumors.json', 'w'))          # clean attributes (+ fixed noise draws)
    ids = {s: sorted(rm.bdmap_id(c) for c in splits[s]) for s in ['test', 'val', 'mask_train', 'report_only']}
    for s, v in ids.items():
        pd.DataFrame({'BDMAP ID': v}).to_csv(f'{META}/ids_{s}.csv', index=False)
    pd.DataFrame([dict(msd_case=r['case'], **{'BDMAP ID': r['bdmap']}, split=r['split']) for r in records]
                 ).sort_values('BDMAP ID').to_csv(f'{META}/id_map.csv', index=False)
    pd.DataFrame(rm.per_ct_metadata(records)).to_csv(f'{META}/per_ct_metadata.csv', index=False)
    ana_all = []
    for sg in SIGMAS:
        rows, ana = rm.report_rows(records, sg, ids=set(ids['report_only']))     # training reports: report_only cases
        pd.DataFrame(rows, columns=rm.REPORT_COLUMNS).to_csv(f'{META}/reports_sigma{sg:.2f}.csv', index=False)
        rows_all, ana2 = rm.report_rows(records, sg)                             # all cases (analysis only)
        pd.DataFrame(rows_all, columns=rm.REPORT_COLUMNS).to_csv(f'{META}/reports_all_sigma{sg:.2f}.csv', index=False)
        ana_all += ana2
    pd.DataFrame(ana_all).to_csv(f'{META}/report_analysis.csv', index=False)
    print(pd.read_csv(f'{META}/reports_sigma{TRAIN_SIGMA:.2f}.csv').head(8).to_string())

### 1d. Quality checks
* Head/body/tail split, tumour channel overlay, and the npz value range.
* **How good is the report-derived volume?** At sigma = 0, the gap between `report_volume_mm3` and `true_volume_mm3` is the error that comes from the ellipsoid model alone.
  This is one reason the Volume Loss has a tolerance tau.

In [6]:
if RUN_QC and os.path.exists(f'{META}/clean_tumors.json'):
    import numpy as np, pandas as pd, matplotlib.pyplot as plt
    recs = json.load(open(f'{META}/clean_tumors.json'))
    df = pd.DataFrame([dict(bdmap=r['bdmap'], split=r['split'], shape=r['npz_shape'], n_tumors=sum(t['reported'] for t in r['tumors']),
                            lesion_vox=r['lesion_vox'], panc_lost=r['pancreas_vox_lost_by_body_crop']) for r in recs])
    print(df.groupby('split').agg(n=('bdmap', 'size'), tumours_mean=('n_tumors', 'mean'), lesion_vox_median=('lesion_vox', 'median')))
    print('cases where the body crop removed pancreas voxels:', int((df.panc_lost > 0).sum()))
    ana = pd.read_csv(f'{META}/report_analysis.csv')
    a0 = ana[ana.sigma == 0.0]
    rel = (a0.report_volume_mm3 / a0.true_volume_mm3)
    print(f'report-volume / true-volume at sigma=0: median {rel.median():.2f}, IQR {rel.quantile(.25):.2f}-{rel.quantile(.75):.2f}, '
          f'within +-10%: {((rel-1).abs() <= .1).mean():.0%}')
    loc = pd.Series([t['location'] for r in recs for t in r['tumors'] if t['reported']]).value_counts()
    print('report locations:\n', loc.to_string())

    fig, ax = plt.subplots(2, 3, figsize=(15, 9))
    for row, (folder, cls) in enumerate([('mask_npz', ['pancreas_head', 'pancreas_body', 'pancreas_tail', 'pancreatic_lesion']),
                                         ('report_npz', ['pancreas_head', 'pancreas_body', 'pancreas_tail'])]):
        r = next(r for r in recs if r['folder'] == folder)
        img = np.load(f'{DATA}/{folder}/{r["bdmap"]}.npz')['arr_0']
        gt = np.load(f'{DATA}/{folder}/{r["bdmap"]}_gt.npz')['arr_0']
        names = r['channels']
        cz = int(np.argwhere(gt[names.index('pancreas')]).mean(0)[0])
        ax[row, 0].imshow(img[cz], cmap='gray'); ax[row, 0].set_title(f'{r["bdmap"]} ({folder}) z={cz}')
        over = np.zeros(img[cz].shape + (4,))
        colors = {'pancreas_head': (1, 0, 0, .4), 'pancreas_body': (0, 1, 0, .4), 'pancreas_tail': (0, 0, 1, .4), 'pancreatic_lesion': (1, 1, 0, .8)}
        for c in cls:
            over[gt[names.index(c)][cz] > 0] = colors[c]
        ax[row, 1].imshow(img[cz], cmap='gray'); ax[row, 1].imshow(over); ax[row, 1].set_title('head=red body=green tail=blue lesion=yellow')
        ax[row, 2].hist(img.ravel()[::50], bins=100); ax[row, 2].set_title(f'npz intensities, shape {img.shape}')
    plt.tight_layout(); plt.show()
    plt.figure(figsize=(5, 5)); plt.loglog(a0.true_volume_mm3, a0.report_volume_mm3, '.', alpha=.5)
    lim = [a0.true_volume_mm3.min(), a0.true_volume_mm3.max()]; plt.plot(lim, lim, 'k--')
    plt.xlabel('true tumour volume (mm^3)'); plt.ylabel('report ellipsoid volume (mm^3), sigma=0'); plt.show()
    if shutil.which('du'):
        print(subprocess.run(['du', '-sh', DATA, f'{DATA}/mask_npz', f'{DATA}/report_npz', f'{DATA}/eval_npz'], capture_output=True, text=True).stdout)

### 1e. Persistence
`/kaggle/working` survives between sessions only when *Persistence = Files only* is set, or when you commit a version and attach its output as an input.
`find_data_root()` in the config cell also searches `/kaggle/input`. As an optional backup, the cell below pushes `rsuper_data` to a **private** Hugging Face dataset repo.
The data are derived from MSD (CC-BY-SA 4.0); keep the repo private.

In [7]:
HF_DATA_REPO = None   # e.g. '<your-hf-username>/rsuper-msd-processed' (private)
if PUSH_DATA_TO_HF and HF_DATA_REPO:
    from huggingface_hub import HfApi
    api = HfApi(token=os.environ.get('HF_TOKEN'))
    api.create_repo(HF_DATA_REPO, repo_type='dataset', private=True, exist_ok=True)
    api.upload_large_folder(repo_id=HF_DATA_REPO, repo_type='dataset', folder_path=DATA)
# To restore in a new session:
#   from huggingface_hub import snapshot_download
#   snapshot_download(HF_DATA_REPO, repo_type='dataset', local_dir=f'{WORK}/rsuper_data'); DATA = f'{WORK}/rsuper_data'

## 2. Training with the repo's `train_ddp.py`

**Config edits** (the originals are kept as `*.yaml.orig`). `train_ddp.py` reads `config/<dataset>/medformer_3d.yaml`, and `predict_abdomenatlas.py` reads `config/abdomenatlas/...`:
* `training_size/window_size = 96^3`: the README suggests 96 or 64 when GPU memory is short; MedFormer at 128 needs more than 30 GB.
* `iter_per_epoch`: this is only a config key. The paper's value is 1000.
* `classes: 5`: the inference script allocates `args.classes` output channels.
* `k_fold: 1`: `train_ddp.py` writes `cross_validation.txt` by looping over `range(k_fold)` while training only fold 0. With the default of 10 it raises `IndexError` *after* training finishes.
* `num_workers`, `print_freq`.

**Stages** (see the README "Train" section):
| run | dataset (loader) | loss flags | meaning |
|---|---|---|---|
| `baseline` | `abdomenatlas` (masks only) | `--report_volume_loss_basic 0` | standard segmentation from 50 masks |
| `control` | `abdomenatlas_ufo` | `--report_volume_loss_basic 0` | same data, iterations and starting point as the report runs, but **no report loss**. It isolates the effect of the report losses (not in the paper) |
| `volume` | `abdomenatlas_ufo` | `--loss dice --report_volume_loss_basic 0.1` | + Volume Loss (`volume_loss_basic`) on every deep-supervision output |
| `volume_ball` | `abdomenatlas_ufo` | `--loss ball_dice_last --report_volume_loss_basic 0.1` | paper default: Ball Loss on the last output, Volume Loss on the deep-supervision outputs |

In [8]:
import yaml, re

def patch_configs(crop=CROP, iters=ITERS_PER_EPOCH, workers=WORKERS):
    for ds in ['abdomenatlas', 'abdomenatlas_ufo']:
        p = f'{RS}/config/{ds}/medformer_3d.yaml'
        if not os.path.exists(p + '.orig'):
            shutil.copy(p, p + '.orig')
        cfg = yaml.safe_load(open(p + '.orig'))
        cfg.update(training_size=[crop] * 3, window_size=[crop] * 3, iter_per_epoch=iters, classes=5,
                   k_fold=1, num_workers=workers, print_freq=25)
        yaml.safe_dump(cfg, open(p, 'w'), sort_keys=False)
    print('patched configs: crop', crop, 'iters/epoch', iters)

class GpuMonitor(threading.Thread):
    # polls nvidia-smi; .peak = max MiB used on any GPU
    def __init__(self, every=3):
        super().__init__(daemon=True); self.every, self.peak, self.stop_flag = every, 0, False
    def run(self):
        while not self.stop_flag:
            out = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                                 capture_output=True, text=True).stdout.split()
            self.peak = max([self.peak] + [int(x) for x in out if x.isdigit()]); time.sleep(self.every)

_port = [9100]
def ckpt_path(dataset, name):
    return f'{EXP}{dataset}/{name}/fold_0_latest.pth'

def ckpt_epoch(path):
    import torch
    return torch.load(path, map_location='cpu', weights_only=False)['epoch'] if os.path.exists(path) else 0

def run_train(name, dataset, extra, epochs, echo_every=50):
    # train_ddp.py in a subprocess; resumes from fold_0_latest.pth after a crash or a new session
    # (same idea as the repo's train_on_merlin.sh restart loop)
    ck = ckpt_path(dataset, name)
    log = f'{LOGS}/{name}.log'
    for attempt in range(MAX_RESTARTS + 1):
        if ckpt_epoch(ck) >= epochs:
            print(f'[{name}] done: {ck} at epoch {ckpt_epoch(ck)}'); break
        _port[0] += 1
        cmd = [sys.executable, 'train_ddp.py', '--dataset', dataset, '--model', 'medformer', '--dimension', '3d',
               '--batch_size', str(BATCH), '--unique_name', name, '--crop_on_tumor', '--all_train', '--gpu', GPUS,
               '--workers', str(WORKERS), '--epochs', str(epochs), '--classes_number', '5', '--crop_size', str(CROP),
               '--cp_path', EXP, '--log_path', f'{WORK}/tb/', '--dist_url', f'tcp://127.0.0.1:{_port[0]}'] + extra
        if os.path.exists(ck):
            cmd += ['--resume', '--load', ck]
        print(f'[{name}] attempt {attempt}:', ' '.join(cmd[1:]))
        t0 = time.time()
        with open(log, 'a') as lf, subprocess.Popen(cmd, cwd=RS, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
            for i, line in enumerate(p.stdout):
                lf.write(line)
                if ('Starting epoch' in line or 'Traceback' in line or 'Error' in line[:200] or i % echo_every == 0) \
                        and 'Chosen segment' not in line and 'Saved to' not in line:
                    print(line.rstrip()[:300])
        print(f'[{name}] exit {p.returncode} after {(time.time()-t0)/60:.1f} min; checkpoint epoch {ckpt_epoch(ck)}')
    for f in glob.glob(f'{EXP}{dataset}/{name}/fold_0_epoch_*.pth'):   # keep disk < 20 GB: only 'latest' is kept
        os.remove(f)
    return ck

def report_crop_stats(name):
    # how often the report loader cropped on the report's sub-segment (otherwise random crop, so no report loss)
    txt = open(f'{LOGS}/{name}.log').read()
    ok, bad = txt.count('Crop on tumor successful'), txt.count('Error cropping around tumor')
    rnd = txt.count('Random crop')
    print(f'[{name}] report crops: on-segment {ok}, failed->random {bad}, random(by chance/empty) {rnd}')

def finetune_args(loss, w, sigma=TRAIN_SIGMA):
    return ['--pretrain', '--pretrained', ckpt_path('abdomenatlas', 'baseline'),
            '--loss', loss, '--report_volume_loss_basic', str(w), '--volume_loss_tolerance', str(VOLUME_TOL),
            '--data_root', MASK_NPZ, '--UFO_root', REPORT_NPZ, '--ucsf_ids', f'{META}/ids_report_only.csv',
            '--reports', f'{META}/reports_sigma{sigma:.2f}.csv', '--tumor_classes', 'pancreas', '--lr', str(FT_LR)]

FINETUNES = {'control': ('dice', 0.0), 'volume': ('dice', REPORT_W), 'volume_ball': ('ball_dice_last', REPORT_W)}
BASE_ARGS = ['--data_root', MASK_NPZ, '--report_volume_loss_basic', '0', '--lr', str(BASE_LR)]

### 2a. Probe: speed and memory (run once on the GPU)
This runs one epoch of 20 iterations for the baseline and for `volume_ball`. It prints seconds per iteration, peak GPU memory, and the projected hours for the epoch settings in the config cell.
If memory exceeds about 14.5 GB, set `CROP = 64`. If `volume_ball` is much slower, the cost is the Ball Loss's ball convolution (`isolate_tumor`: a `conv3d` with a kernel as wide as the tumour).

In [9]:
if RUN_PROBE:
    patch_configs(iters=20)
    res = {}
    for name, ds, extra in [('probe_baseline', 'abdomenatlas', BASE_ARGS),
                            ('probe_volume_ball', 'abdomenatlas_ufo', None)]:
        if extra is None:   # fine-tune probe starts from the probe baseline
            extra = finetune_args('ball_dice_last', REPORT_W)
            extra[extra.index('--pretrained') + 1] = ckpt_path('abdomenatlas', 'probe_baseline')
        shutil.rmtree(f'{EXP}{ds}/{name}', ignore_errors=True)
        open(f'{LOGS}/{name}.log', 'w').close()
        mon = GpuMonitor(); mon.start(); t0 = time.time()
        run_train(name, ds, extra, epochs=1, echo_every=10)
        mon.stop_flag = True
        res[name] = dict(sec_per_iter=(time.time() - t0) / 20, peak_gpu_mib=mon.peak)
    for k, v in res.items():
        print(k, v)
    print('NOTE: s/iter includes start-up (~1-2 min), so the projection is pessimistic.')
    print(f'projected baseline: {res["probe_baseline"]["sec_per_iter"]*ITERS_PER_EPOCH*BASE_EPOCHS/3600:.1f} h; '
          f'each fine-tune: {res["probe_volume_ball"]["sec_per_iter"]*ITERS_PER_EPOCH*FT_EPOCHS/3600:.1f} h')
    report_crop_stats('probe_volume_ball')
    patch_configs()

### 2b. Masks-only baseline (50 CT-mask pairs)

In [10]:
if RUN_BASELINE:
    patch_configs()
    run_train('baseline', 'abdomenatlas', BASE_ARGS, epochs=BASE_EPOCHS)

patched configs: crop 96 iters/epoch 200
[baseline] attempt 0: train_ddp.py --dataset abdomenatlas --model medformer --dimension 3d --batch_size 2 --unique_name baseline --crop_on_tumor --all_train --gpu 0,1 --workers 2 --epochs 50 --classes_number 5 --crop_size 96 --cp_path /kaggle/working/exp/ --log_path /kaggle/working/tb/ --dist_url tcp://127.0.0.1:9101 --data_root /kaggle/working/rsuper_data/mask_npz --report_volume_loss_basic 0 --lr 0.001
Traceback (most recent call last):
FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/rsuper_data/mask_npz/list/dataset.yaml'
[baseline] exit 1 after 0.8 min; checkpoint epoch 0
[baseline] attempt 1: train_ddp.py --dataset abdomenatlas --model medformer --dimension 3d --batch_size 2 --unique_name baseline --crop_on_tumor --all_train --gpu 0,1 --workers 2 --epochs 50 --classes_number 5 --crop_size 96 --cp_path /kaggle/working/exp/ --log_path /kaggle/working/tb/ --dist_url tcp://127.0.0.1:9102 --data_root /kaggle/working/rsup

### 2c. Report supervision (the masks plus 150 CT-report pairs), each run starting from the baseline
In `training/dataset/dim3/dataset_abdomenatlas_UFO.py`, a report case is processed like this:
`get_tumor_segment_labels` reads the report rows, then `crop` places the crop around the reported sub-segment (`crop_foreground_3d`), `assign_labels` marks the tumour channel there as unknown,
and `estimate_tumor_volume` sums the report volumes of the tumours inside that sub-segment. These go to `calculate_loss` as `chosen_segment_mask`, `unk_voxels` and `tumor_volumes_report`.

In [11]:
for name in RUN_FINETUNE:
    assert os.path.exists(ckpt_path('abdomenatlas', 'baseline')), 'train the baseline first'
    patch_configs()
    loss, w = FINETUNES[name]
    run_train(name, 'abdomenatlas_ufo', finetune_args(loss, w), epochs=FT_EPOCHS)
    report_crop_stats(name)

## 3. Evaluation on the 50 held-out MSD test CTs
The repo's `predict_abdomenatlas.py` runs sliding-window inference on the npz test volumes, which are body-cropped with no label used.
Its two-stage mode targets other organs, so we pass `--disable_inference_2_stages`. Two processes run at once, one per T4.
Metrics are computed on the 1 mm grid:
* **Dice** for `pancreatic_lesion` per CT (mean ± sd), and for `pancreas`.
* **Lesion-level detection**: a GT tumour (≥ 50 mm^3) counts as detected if any predicted voxel overlaps it, and a predicted component (≥ 50 mm^3) that touches no GT tumour is a false positive.
  From these we report precision, recall and **F1**.
* **Patient-level sensitivity**: predicted tumour volume ≥ 50 mm^3. Specificity is not available because MSD has no healthy CTs.
* Each metric is given without and with the repo's `--organ_mask_on_lesion` post-processing (keep tumour only inside the dilated *predicted* pancreas).

In [12]:
import numpy as np, pandas as pd, yaml
from scipy import ndimage as ndi

MODELS = {'baseline': 'abdomenatlas', 'control': 'abdomenatlas_ufo', 'volume': 'abdomenatlas_ufo', 'volume_ball': 'abdomenatlas_ufo'}

def predict(name, split='test'):
    ck = ckpt_path(MODELS[name], name)
    assert os.path.exists(ck), ck
    patch_configs()
    procs = []
    gpus = GPUS.split(',')
    for k, g in enumerate(gpus):
        cmd = [sys.executable, 'predict_abdomenatlas.py', '--load', ck, '--img_path', EVAL_NPZ,
               '--class_list', f'{MASK_NPZ}/list/label_names.yaml', '--save_path', f'{RESULTS}/{split}',
               '--ids', f'{META}/ids_{split}.csv', '--gpu', g, '--disable_inference_2_stages',
               '--parts', str(len(gpus)), '--current_part', str(k)]
        procs.append(subprocess.Popen(cmd, cwd=RS, stdout=open(f'{LOGS}/predict_{name}_{split}_{k}.log', 'w'), stderr=subprocess.STDOUT))
    for p in procs:
        p.wait()
    print(name, 'predict exit codes', [p.returncode for p in procs])
    return f'{RESULTS}/{split}/abdomenatlas/{name}'   # predict_abdomenatlas.py appends <dataset>/<ckpt folder name>

def components(mask, min_vox):
    cc, n = ndi.label(mask, structure=np.ones((3, 3, 3), bool))
    sizes = ndi.sum(mask, cc, range(1, n + 1)) if n else []
    return cc, [i + 1 for i, s in enumerate(sizes) if s >= min_vox]

def evaluate(name, split='test', min_vox=MIN_TUMOR_VOX):
    import nibabel as nib
    pred_root = f'{RESULTS}/{split}/abdomenatlas/{name}'
    classes = sorted(yaml.safe_load(open(f'{MASK_NPZ}/list/label_names.yaml')))
    rows = []
    for bd in pd.read_csv(f'{META}/ids_{split}.csv')['BDMAP ID']:
        gt = np.load(f'{EVAL_NPZ}/{bd}_gt.npz')['arr_0']
        les_gt, panc_gt = gt[classes.index('pancreatic_lesion')] > 0, gt[classes.index('pancreas')] > 0
        les_p = np.asarray(nib.load(f'{pred_root}/{bd}/predictions/pancreatic_lesion.nii.gz').dataobj) > 0.5
        panc_p = np.asarray(nib.load(f'{pred_root}/{bd}/predictions/pancreas.nii.gz').dataobj) > 0.5
        assert les_p.shape == les_gt.shape, (bd, les_p.shape, les_gt.shape)
        organ = ndi.binary_dilation(panc_p, structure=np.ones((3, 3, 3)))     # as postprocess_npz(--organ_mask_on_lesion)
        for variant, lp in [('raw', les_p), ('organ_masked', les_p & organ)]:
            inter = (lp & les_gt).sum()
            dice = 2 * inter / (lp.sum() + les_gt.sum()) if (lp.sum() + les_gt.sum()) else 1.0
            ccg, gid = components(les_gt, min_vox)
            ccp, pid = components(lp, min_vox)
            tp = sum(lp[ccg == i].any() for i in gid)
            fp = sum(not les_gt[ccp == j].any() for j in pid)
            rows.append(dict(model=name, case=bd, variant=variant, dice_lesion=dice,
                             dice_pancreas=2 * (panc_p & panc_gt).sum() / max(1, panc_p.sum() + panc_gt.sum()),
                             gt_lesions=len(gid), tp=tp, fp=fp, pred_vol=int(lp.sum()), gt_vol=int(les_gt.sum()),
                             patient_detected=int(lp.sum() >= min_vox)))
    return pd.DataFrame(rows)

def summarize(df):
    g = df.groupby(['model', 'variant'])
    s = g.agg(dice_lesion_mean=('dice_lesion', 'mean'), dice_lesion_sd=('dice_lesion', 'std'),
              dice_pancreas=('dice_pancreas', 'mean'), tp=('tp', 'sum'), fp=('fp', 'sum'),
              n_gt_lesions=('gt_lesions', 'sum'), patient_sens=('patient_detected', 'mean'), n=('case', 'nunique'))
    s['lesion_recall'] = s['tp'] / s['n_gt_lesions'].clip(lower=1)
    s['lesion_precision'] = s['tp'] / (s['tp'] + s['fp']).clip(lower=1)
    s['lesion_F1'] = 2 * s['lesion_precision'] * s['lesion_recall'] / (s['lesion_precision'] + s['lesion_recall']).clip(lower=1e-9)
    return s.round(3)

all_rows = []
for name in RUN_EVAL:
    predict(name, 'test')
    df = evaluate(name, 'test')
    df.to_csv(f'{RESULTS}/metrics_{name}_test.csv', index=False)
    all_rows.append(df)
done = [pd.read_csv(f) for f in sorted(glob.glob(f'{RESULTS}/metrics_*_test.csv'))]
if done:
    summary = summarize(pd.concat(done))
    summary.to_csv(f'{RESULTS}/summary_test.csv')
    display(summary)

## 4. Extension (to be defined)
The synthetic reports allow a **controlled report-noise sweep**: `meta/reports_sigma{0.00,0.10,0.20,0.30}.csv` all use the same per-tumour draws `z`, so the noise level is the only thing that changes.
Example run: `run_train('volume_ball_s030', 'abdomenatlas_ufo', finetune_args('ball_dice_last', REPORT_W, sigma=0.30), epochs=FT_EPOCHS)`.
To evaluate it, add `MODELS['volume_ball_s030'] = 'abdomenatlas_ufo'` first.
Your extension idea (adaptive tau, a soft Ball Loss, confidence weighting, ...) would plug into `training/losses_foundation.py` and be selected through `--loss`.

In [13]:
# placeholder for the extension runs